# 00 — Preparacao do dataset (ASDID - Auburn Soybean Disease Image Dataset)

Organiza as imagens brutas (uma subpasta por classe) em **splits estratificados 70/15/15** salvos como CSV, e faz uma EDA rapida (contagem por classe, amostras visuais).

**Classes (6):** cercosporiose, ferrugem-asiatica, mancha-alvo, mancha-olho-de-ra (ex-antracnose), mildio, saudavel.

**Fonte:** ASDID (Zenodo 7304859, licenca CC0). *frogeye* (mancha-olho-de-ra) substitui antracnose, ausente no ASDID (ver TCC-093).

**Entrada esperada:** `RAW_DIR/<classe>/*.jpg`. **Saida:** `data/processed/{train,val,test}.csv` + `label_map.csv` (consumidos pelos notebooks de treino).

## 0. Ambiente (Colab)

In [ ]:
import sys
from pathlib import Path

# Em Colab: instala dependencias, clona o repo e (opcional) monta o Drive
if 'google.colab' in sys.modules:
    !pip install -q timm albumentations scikit-learn pandas matplotlib seaborn tensorboard tqdm pyyaml onnx onnxruntime
    !git clone https://github.com/SEU_USUARIO/tcc-ze-praga-model-playground.git
    %cd tcc-ze-praga-model-playground
    from google.colab import drive
    drive.mount('/content/drive')

print('Setup OK')

## 1. Imports e caminhos

In [ ]:
import sys, csv
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, str(Path('.').resolve()))
from src.data.splits import generate_splits

CLASSES = ['cercosporiose', 'ferrugem-asiatica', 'mancha-alvo', 'mancha-olho-de-ra', 'mildio', 'saudavel']

# Ajuste RAW_DIR para onde estao as imagens brutas (subpasta por classe).
# No Colab/Drive, ex.: Path('/content/drive/MyDrive/ze-praga-dataset/raw')
RAW_DIR = Path('data/raw/asdid')
OUT_DIR = Path('data/processed')
print('RAW_DIR:', RAW_DIR)
print('OUT_DIR:', OUT_DIR)

## 2. Cobertura por classe

Conta as imagens por classe. **Importante (ADR-0001):** se alguma classe (ex.: *cercosporiose*) tiver poucas imagens (~ < 100), considere substitui-la e registrar um ADR — ver TCC-093.

In [ ]:
EXT = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}

counts = {}
for cls in CLASSES:
    d = RAW_DIR / cls
    counts[cls] = len([p for p in d.iterdir() if p.suffix in EXT]) if d.exists() else 0

print('Total de imagens:', sum(counts.values()))
for cls, n in counts.items():
    flag = '  <-- COBERTURA BAIXA' if 0 < n < 100 else ('  <-- AUSENTE' if n == 0 else '')
    print('  ' + cls.ljust(18) + ': ' + str(n) + flag)

plt.figure(figsize=(8, 4))
plt.bar(list(counts.keys()), list(counts.values()), color='#2D6A4F')
plt.xticks(rotation=30, ha='right')
plt.title('Imagens por classe (bruto)')
plt.tight_layout()
plt.show()

## 3. Integridade (imagens corrompidas)

In [ ]:
corrupted = []
for cls in CLASSES:
    d = RAW_DIR / cls
    if not d.exists():
        continue
    for p in d.iterdir():
        if p.suffix not in EXT:
            continue
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception as e:
            corrupted.append((str(p), str(e)))

print('Corrompidas:', len(corrupted))
for path, err in corrupted[:20]:
    print(' ', path, '->', err)

## 4. Gerar splits estratificados (70/15/15) em CSV

Usa `generate_splits` (o mesmo codigo do treino), garantindo CSVs compativeis com `SoybeanLeafDataset`.

In [ ]:
generate_splits(RAW_DIR, OUT_DIR)

for split in ['train', 'val', 'test']:
    df = pd.read_csv(OUT_DIR / (split + '.csv'))
    print(split, '->', len(df), 'amostras |', df['label'].value_counts().to_dict())

## 5. Amostras visuais

In [ ]:
train_df = pd.read_csv(OUT_DIR / 'train.csv')
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, cls in zip(axes.flatten(), CLASSES):
    sub = train_df[train_df['label'] == cls]
    if len(sub) == 0:
        ax.axis('off')
        continue
    img = Image.open(sub.iloc[0]['filepath']).convert('RGB').resize((224, 224))
    ax.imshow(img)
    ax.set_title(cls, fontsize=10)
    ax.axis('off')
plt.suptitle('Uma amostra por classe (train)')
plt.tight_layout()
plt.show()

## 6. Pronto

In [ ]:
print('CSVs gerados em:', OUT_DIR.resolve())
print('Arquivos:', sorted(p.name for p in OUT_DIR.glob('*.csv')))
print('Proximo passo: 01_train_resnet50 (treino).')